# Using PYNQ library for PMOD_ADC

This just uses the built in Pmod_ADC library to read the value on the PMOD_AD2 peripheral.

In [1]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import Pmod_ADC
base = BaseOverlay("base.bit")

In [2]:
adc = Pmod_ADC(base.PMODA)

Read the raw value and the 12 bit values from channel 1.

Refer to docs: https://pynq.readthedocs.io/en/v2.1/pynq_package/pynq.lib/pynq.lib.pmod.html#pynq-lib-pmod

In [3]:
adc.read_raw(ch1=1, ch2=0, ch3=0)

[768]

In [4]:
adc.read(ch1=1, ch2=0, ch3=0)

[1.5024, 0.0049, 0.0]

# Using MicroblazeLibrary

Here we're going down a level and using the microblaze library to write I2C commands directly to the PMOD_AD2 peripheral

Use the documentation on the PMOD_AD2 to answer lab questions

In [30]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
base = BaseOverlay("base.bit")

In [31]:
liba = MicroblazeLibrary(base.PMODA, ['i2c'])

In [32]:
dir(liba) # list the available commands for the liba object

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_build_constants',
 '_build_functions',
 '_mb',
 '_populate_typedefs',
 '_rpc_stream',
 'active_functions',
 'i2c_close',
 'i2c_get_num_devices',
 'i2c_open',
 'i2c_open_device',
 'i2c_read',
 'i2c_write',
 'release',
 'reset',
 'visitor']

In the cell below, open a new i2c device. Check the resources for the i2c_open parameters

In [33]:
device = liba.i2c_open(3,2) # TODO open a device

In [34]:
dir(device) # list the commands for the device class

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__index__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_call_func',
 '_file',
 '_val',
 'close',
 'read',
 'write']

Below we write a command to the I2C channel and then read from the I2C channel. Change the buf[0] value to select different channels. See the AD spec sheet Configuration Register. https://www.analog.com/media/en/technical-documentation/data-sheets/AD7991_7995_7999.pdf

Changing the number of channels to read from will require a 2 byte read for each channel!

In [62]:

buf = bytearray(2)
buf[0] = 0b11110000
device.write(0x28, buf, 1)
device.read(0x28, buf, 2)
data = int(((buf[0] << 8) | buf[1]))
print(format(data, '#018b'))
print(format(data, '#06x'))
print(data)

device.read(0x28, buf, 2)
print(buf[0]<<8 | buf[1] )
device.read(0x28, buf, 2)
print(buf[0]<<8 | buf[1] )
device.read(0x28, buf, 2)
print(buf[0]<<8 | buf[1] )

0b0000000000000000
0x0000
0
4102
8198
12288


In [76]:
def read_all(dev):
    buf = bytearray(2)
    buf[0] = 0b11110000
    device.write(0x28, buf, 1)
    for i in range(4):
        device.read(0x28, buf, 2)
        data = int(((buf[0]&0x0F << 8) | buf[1]))
        chan = int(buf[0]>>4)
        print(f"Chan: {chan} reading Raw: {data:03} Converted: {3.3*data/255}V")
read_all(device)

Chan: 0 reading Raw: 000 Converted: 0.0V
Chan: 1 reading Raw: 255 Converted: 3.3V
Chan: 2 reading Raw: 029 Converted: 0.3752941176470588V
Chan: 3 reading Raw: 001 Converted: 0.012941176470588235V


Compare the binary output given by ((buf[0]<<8) | buf[1]) to the AD7991 spec sheet. You can select the data only using the following command

In [20]:
result_12bit = (((buf[0] & 0x0F) << 8) | buf[1])

# Using MicroBlaze

In [26]:
base = BaseOverlay("base.bit")

In [28]:
%%microblaze base.PMODA

#include "i2c.h"

int read_adc(){
    i2c i2c_device = i2c_open(3, 2);
    unsigned char buf[2];
    buf[0] = 0;
    i2c_write(i2c_device, 0x28, buf, 1);
    i2c_read(i2c_device, 0x28, buf, 2);
    return ((buf[0] & 0x0F) << 8) | buf[1];
}

In [29]:
read_adc()

271